# Silver → Gold: Enrich and Export to Parquet

This notebook reads **already cleaned** piece data from the silver layer (`silver.clean_pieces`), enriches it with computed features, and exports to a parquet file ready for analytics and ML.

### What happens here (enrichment only — no cleaning)

All data quality cleaning was done in the bronze → silver step. This notebook only adds analytical value:

1. Compute partial times between process stages (inter-stage durations)
2. Mark production gaps and assign production run IDs
3. Export to `data/gold/pieces.parquet`

### Why this regenerates fully

Production run IDs depend on the ordering of all pieces globally. Adding new data changes the run boundaries, so the gold output is always rebuilt from the complete silver table.

In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# -----------------------------
# Load environment variables
# -----------------------------
env_path = Path("../infra/.env")
if not env_path.exists():
    raise FileNotFoundError(f"Could not find env file: {env_path.resolve()}")

env = {}
for line in env_path.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    k, v = line.split("=", 1)
    env[k.strip()] = v.strip()

required = ["POSTGRES_PORT", "POSTGRES_DB", "POSTGRES_USER", "POSTGRES_PASSWORD"]
missing = [k for k in required if k not in env]
if missing:
    raise ValueError(f"Missing keys in ../infra/.env: {missing}")

# -----------------------------
# Database connection
# -----------------------------
engine = create_engine(
    f"postgresql+psycopg2://{env['POSTGRES_USER']}:{env['POSTGRES_PASSWORD']}"
    f"@localhost:{env['POSTGRES_PORT']}/{env['POSTGRES_DB']}"
)

with engine.connect() as conn:
    print("Connected to PostgreSQL:", conn.execute(text("SELECT current_database()")).scalar())

# -----------------------------
# Output path
# -----------------------------
gold_dir = Path("../data/gold")
gold_dir.mkdir(parents=True, exist_ok=True)

gold_path = gold_dir / "pieces.parquet"
print("Gold output path:", gold_path.resolve())



Connected to PostgreSQL: vaultech
Gold output path: C:\Users\warre\VaultTech\data\gold\pieces.parquet


## Load silver data

Read the full `silver.clean_pieces` table. All cleaning (zeros, upsetting, outliers, monotonic order) was already applied.

In [15]:
gold = pd.read_sql("""
SELECT *
FROM silver.clean_pieces
ORDER BY timestamp
""", engine)

gold["timestamp"] = pd.to_datetime(gold["timestamp"], utc=True, errors="coerce")

print("silver.clean_pieces shape:", gold.shape)
print("columns:", gold.columns.tolist())
display(gold.head(10))



silver.clean_pieces shape: (140404, 11)
columns: ['timestamp', 'piece_id', 'die_matrix', 'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_bath_s', 'lifetime_general_s', 'processed_at', 'oee_cycle_time_s', 'lifetime_auxiliary_press_s']


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_bath_s,lifetime_general_s,processed_at,oee_cycle_time_s,lifetime_auxiliary_press_s
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,56.200001,56.200001,2026-04-14 02:34:28.839803+00:00,NaN,54.599998
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,56.400002,56.400002,2026-04-14 02:34:28.839803+00:00,NaN,54.799999
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,56.900002,56.900002,2026-04-14 02:34:28.839803+00:00,NaN,55.299999
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,57.099998,57.099998,2026-04-14 02:34:28.839803+00:00,NaN,55.500000
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,57.200001,57.200001,2026-04-14 02:34:28.839803+00:00,NaN,55.500000
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,55.200001,55.200001,2026-04-14 02:34:28.839803+00:00,13.4690,53.599998
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,56.400002,56.400002,2026-04-14 02:34:28.839803+00:00,13.4496,54.700001
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,56.299999,56.299999,2026-04-14 02:34:28.839803+00:00,13.2704,54.700001
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,23.299999,36.599998,54.900002,54.900002,2026-04-14 02:34:28.839803+00:00,NaN,53.299999
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,23.600000,36.799999,55.799999,55.799999,2026-04-14 02:34:28.839803+00:00,NaN,54.099998


## Step 1: Compute partial times between stages

Since the lifetime columns are cumulative (each includes all previous stages), we compute the time spent **between consecutive stages** by subtraction. All values in **seconds**.

| Partial column | Calculation | What it measures |
|---|---|---|
| `partial_furnace_to_2nd_strike_s` | `lifetime_2nd_strike_s` | Robot pick at furnace, transfer to main press, positioning |
| `partial_2nd_to_3rd_strike_s` | `lifetime_3rd - lifetime_2nd` | Press retraction, robot repositioning between strikes |
| `partial_3rd_to_4th_strike_s` | `lifetime_4th - lifetime_3rd` | Transfer within main press to drill station |
| `partial_4th_strike_to_auxiliary_press_s` | `lifetime_aux - lifetime_4th` | Exit main press, transfer to auxiliary press, deburring and coining |
| `partial_auxiliary_press_to_bath_s` | `lifetime_bath - lifetime_aux` | Transport from auxiliary press to quench bath |

These are the key diagnostic values: when a piece is slow, the partial that deviates identifies which segment caused the delay.

In [16]:
gold = gold.sort_values("timestamp").reset_index(drop=True).copy()

# Partial times between consecutive process stages
gold["partial_furnace_to_2nd_strike_s"] = gold["lifetime_2nd_strike_s"]

gold["partial_2nd_to_3rd_strike_s"] = (
    gold["lifetime_3rd_strike_s"] - gold["lifetime_2nd_strike_s"]
)

gold["partial_3rd_to_4th_strike_s"] = (
    gold["lifetime_4th_strike_s"] - gold["lifetime_3rd_strike_s"]
)

gold["partial_4th_strike_to_auxiliary_press_s"] = (
    gold["lifetime_auxiliary_press_s"] - gold["lifetime_4th_strike_s"]
)

gold["partial_auxiliary_press_to_bath_s"] = (
    gold["lifetime_bath_s"] - gold["lifetime_auxiliary_press_s"]
)

partial_cols = [
    "partial_furnace_to_2nd_strike_s",
    "partial_2nd_to_3rd_strike_s",
    "partial_3rd_to_4th_strike_s",
    "partial_4th_strike_to_auxiliary_press_s",
    "partial_auxiliary_press_to_bath_s",
]

display(
    gold[
        [
            "timestamp",
            "piece_id",
            "die_matrix",
            "lifetime_2nd_strike_s",
            "lifetime_3rd_strike_s",
            "lifetime_4th_strike_s",
            "lifetime_auxiliary_press_s",
            "lifetime_bath_s",
        ] + partial_cols
    ].head(10)
)

display(
    gold[["die_matrix"] + partial_cols]
    .groupby("die_matrix")
    .median()
    .round(2)
    .reset_index()
)

,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,17.900000,6.700001,13.400000,16.599998,1.600002
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,17.900000,6.700001,13.300001,16.899998,1.600002
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,18.200001,6.599998,13.500000,17.000000,1.600002
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,18.400000,6.700001,13.300001,17.099998,1.599998
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,18.200001,6.599998,13.400002,17.299999,1.700001
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,53.599998,55.200001,16.700001,6.699999,13.400000,16.799999,1.600002
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,54.700001,56.400002,18.299999,6.600000,13.300001,16.500000,1.700001
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,54.700001,56.299999,17.900000,6.600000,13.200001,17.000000,1.599998
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,23.299999,36.599998,53.299999,54.900002,16.700001,6.599998,13.299999,16.700001,1.600002
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,23.600000,36.799999,54.099998,55.799999,16.700001,6.900000,13.199999,17.299999,1.700001


,die_matrix,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s
0,4974,17.3,6.5,13.1,17.0,1.8
1,5052,18.3,6.9,13.7,17.3,1.6
2,5090,17.7,6.8,13.8,17.7,1.6
3,5091,17.9,6.6,13.5,17.0,1.6


## Step 2: Mark production gaps

Flag pieces that follow a production gap (> 5 minutes since the previous piece). Assign a `production_run_id` that groups consecutive pieces within the same uninterrupted production run.

This prevents time-series analysis from interpolating across machine stops, weekends, or maintenance periods.

In [18]:
gold = gold.sort_values("timestamp").reset_index(drop=True).copy()

# Time since previous piece
gold["seconds_since_prev_piece"] = gold["timestamp"].diff().dt.total_seconds()

# A production gap is any interval > 5 minutes
gold["is_production_gap"] = gold["seconds_since_prev_piece"] > 300
gold["is_production_gap"] = gold["is_production_gap"].fillna(False)

# Consecutive pieces between gaps belong to the same production run
gold["production_run_id"] = gold["is_production_gap"].cumsum() + 1

print("Number of production gaps:", int(gold["is_production_gap"].sum()))
print("Number of production runs:", int(gold["production_run_id"].nunique()))

display(
    gold[
        [
            "timestamp",
            "piece_id",
            "die_matrix",
            "seconds_since_prev_piece",
            "is_production_gap",
            "production_run_id",
        ]
    ].head(20)
)

Number of production gaps: 775
Number of production runs: 776


,timestamp,piece_id,die_matrix,seconds_since_prev_piece,is_production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,NaN,False,1
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,12.708,False,1
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,14.609,False,1
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,12.719,False,1
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,14.000,False,1
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,13.309,False,1
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,12.611,False,1
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,13.713,False,1
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,67.332,False,1
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,67.043,False,1


## Export to parquet

Save the gold dataset. This is the file that analytics notebooks, ML training, and the Streamlit app should consume.

In [19]:
from pathlib import Path

# Ensure directory exists
output_path = Path("data/gold")
output_path.mkdir(parents=True, exist_ok=True)

file_path = output_path / "pieces.parquet"

# Sort columns in process order (important for readability + ML)
cols_order = [
    "timestamp",
    "piece_id",
    "die_matrix",
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
    "partial_furnace_to_2nd_strike_s",
    "partial_2nd_to_3rd_strike_s",
    "partial_3rd_to_4th_strike_s",
    "partial_4th_strike_to_auxiliary_press_s",
    "partial_auxiliary_press_to_bath_s",
    "oee_cycle_time_s",
    "seconds_since_prev_piece",
    "is_production_gap",
    "production_run_id",
]

cols_order = [c for c in cols_order if c in gold.columns]

gold = gold[cols_order].copy()

# Write parquet
gold.to_parquet(file_path, index=False)

print("Parquet file saved to:", file_path)
print("Final gold shape:", gold.shape)


Parquet file saved to: data\gold\pieces.parquet
Final gold shape: (140404, 18)


In [20]:
# Reload parquet to verify integrity
gold_check = pd.read_parquet(file_path)

print("Reloaded shape:", gold_check.shape)
print("Columns match:", list(gold_check.columns) == list(gold.columns))

display(gold_check.head())

Reloaded shape: (140404, 18)
Columns match: True


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s,oee_cycle_time_s,seconds_since_prev_piece,is_production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,17.900000,6.700001,13.400000,16.599998,1.600002,NaN,NaN,False,1
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,17.900000,6.700001,13.300001,16.899998,1.600002,NaN,12.708,False,1
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,18.200001,6.599998,13.500000,17.000000,1.600002,NaN,14.609,False,1
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,18.400000,6.700001,13.300001,17.099998,1.599998,NaN,12.719,False,1
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,18.200001,6.599998,13.400002,17.299999,1.700001,NaN,14.000,False,1


## Summary

## 📊 Summary — Gold dataset

The gold dataset represents the final, analysis-ready version of the forging line data.

All data quality issues were resolved upstream in the silver layer. This step focuses purely on enrichment and structuring the data for downstream consumption.

---

### 🔧 What was added in this step

**Partial times between stages**  
The cumulative lifetime signals were transformed into inter-stage durations.  
These values are critical for diagnosing delays, as they directly indicate which segment of the process is responsible when a piece is slow.

**Production gap detection**  
Gaps greater than 5 minutes between consecutive pieces were identified.  
These correspond to real production interruptions such as maintenance, weekends, or machine stops.

**Production run segmentation**  
A `production_run_id` was assigned to group pieces produced in continuous runs.  
This prevents time-series analysis from incorrectly bridging across production stops.

---

### 📦 Output

The enriched dataset was exported to:


data/gold/pieces.parquet


- ~140k rows (one per valid piece)  
- Columnar, compressed format optimized for analytics and ML  

---